# Temporal Difference (TD) Learning in Gridworld
This notebook demonstrates the basics of Temporal Difference (TD) learning for value estimation in a simple Gridworld environment. Each section is explained below.

**What is Temporal Difference (TD) Learning?**

Temporal Difference learning is a core method in reinforcement learning that updates value estimates for states based on the difference (the "temporal difference") between consecutive predictions. Unlike Monte Carlo methods, which wait until the end of an episode to update values, TD learning updates values after every step using the observed reward and the estimated value of the next state. This allows for faster and more incremental learning.

In this notebook, you will see how TD(0) learning is applied to a 4x4 gridworld: the agent explores the environment, collects rewards, and updates its value estimates for each state using the TD update rule. The notebook visualizes the reward structure, the learning process, and the resulting value function.

In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt

## Environment and Parameters Setup
We define the reward structure for each state in the 4x4 grid, set the terminal state, initialize the state values, learning rate (alpha), and a log for scores.

In [ ]:
rewards = np.zeros(16)
rewards[3] = 10
rewards[2] = -1
rewards[11] = -1
rewards[10] = -1

terminal_state = 3
state_values = np.zeros(16)
alpha = 0.02
score_log = []

## Visualizing the Reward Structure
This cell visualizes the reward values for each state in the 4x4 grid. High and low reward states are easily identified in the plot.

In [ ]:
plt.imshow(rewards.reshape(4, 4))

## Loading the State Transition Table
We load the state transition table from a CSV file. This table defines how the agent moves between states given an action.

In [ ]:
state_transition_table = np.genfromtxt("state_transitions.csv", delimiter=",").astype(int)

## TD(0) Update Function
This function implements the TD(0) update rule, which updates the value estimates for each state based on the observed rewards and the estimated value of the next state.

In [ ]:
def TD_update(next_val, values, rewards, states):
    gamma = 0.9
    next_val = next_val
    #fixed bug in code here
    new_values = np.zeros(16) + values
    for i in reversed(range(len(rewards))):
        new_values[states[i]] = values[states[i]] + alpha * (rewards[i] + gamma * next_val - values[states[i]])
        next_val = values[states[i]]
    return new_values

## Agent Evaluation Function
This function simulates the agent's behavior using the current value estimates to select actions, and returns the total reward and the sequence of visited states.

In [ ]:
def test_agent():
    state = 12
    done = False
    steps = 0
    total_rewards = 0
    states_log = []
    while (not(state == terminal_state)) and steps<30:
        states_log.append(state)
        action = np.argmax(state_values[state_transition_table[state]])
        state = state_transition_table[state, action]
        total_rewards += rewards[state]
        steps += 1
    states_log.append(state)
    return total_rewards, states_log

## TD Learning: Main Training Loop
This loop runs multiple episodes where the agent explores the environment, collects rewards, and updates the value estimates using the TD(0) update rule. The agent's performance is logged for analysis.

In [ ]:
for _ in range(10000):
    state = 12
    state_log = []
    reward_log = []
    steps = 0

    while (not(state == terminal_state)) and steps<30:
        action = random.randint(0,3)
        state_log.append(state)
        reward_log.append(rewards[state])

        state = state_transition_table[state, action]
        steps += 1
        
    state_log.append(state)
    reward_log.append(rewards[state])
    next_val = 0
    state_values = TD_update(next_val, state_values, reward_log, state_log)
    
    score_log.append(test_agent()[0])

## Visualizing Learned State Values
This cell displays the learned value for each state after training, helping you see which states are considered more valuable by the agent.

In [ ]:
fig1, ax1= plt.subplots(1)
ax1.imshow(state_values.reshape(4, 4))

for (j,i), label in np.ndenumerate(state_values.reshape(4, 4).round(3)):
    ax1.text(i,j,label,ha='center',va='center')

## Plotting the Learning Curve
This cell plots the agent's score over time, showing how the agent's performance improves as it learns.

In [ ]:
plt.plot(score_log)

## Visualizing the Agent's Path
This cell shows the path taken by the agent from the start state to the terminal state using the learned value function.

In [ ]:
_, state_log = test_agent()
state_view = np.zeros(16)
state_view[state_log] = 1
plt.imshow(state_view.reshape(4,4))